# 02 — Data preparation and feature engineering

Build a clean Won/Lost modeling table. Raw source files remain unchanged; all normalization is performed on derived keys.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw').exists() and (ROOT.parent / 'data' / 'raw').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'raw'
def find_csv(name):
    for p in [DATA/name, Path.cwd()/name, Path.cwd().parent/'data'/'raw'/name]:
        if p.exists(): return p
    raise FileNotFoundError(f'Missing {name}; expected files under data/raw/.')
pipeline = pd.read_csv(find_csv('sales_pipeline.csv'))
accounts = pd.read_csv(find_csv('accounts.csv'))
products = pd.read_csv(find_csv('products.csv'))
teams = pd.read_csv(find_csv('sales_teams.csv'))
print('Pipeline columns:', pipeline.columns.tolist())
print('Account columns:', accounts.columns.tolist())
print('Product columns:', products.columns.tolist())
print('Team columns:', teams.columns.tolist())

## Build derived joins

The source uses `GTXPro` in the pipeline and `GTX Pro` in the product reference. A derived key reconciles that spelling without rewriting source values.

In [ ]:
pipeline['product_join_key'] = pipeline['product'].replace({'GTXPro': 'GTX Pro'}) if 'product' in pipeline else np.nan
products['product_join_key'] = products['product'] if 'product' in products else np.nan
# Validate reference-key uniqueness before joining.
for label, frame, key in [('accounts', accounts, 'account'), ('products', products, 'product_join_key'), ('teams', teams, 'sales_agent')]:
    if key in frame:
        duplicates = frame[key].duplicated().sum()
        print(f'{label}: duplicate {key} keys = {duplicates}')
        if duplicates:
            raise ValueError(f'{label} reference key {key} is not unique; resolve before joining.')
# Join only when the expected key exists in each table.
model = pipeline.copy()
if 'account' in model and 'account' in accounts:
    model = model.merge(accounts, on='account', how='left', validate='many_to_one', suffixes=('', '_account'))
if 'product_join_key' in model and 'product_join_key' in products:
    model = model.merge(products.drop(columns=['product'], errors='ignore'), on='product_join_key', how='left', validate='many_to_one', suffixes=('', '_product'))
if 'sales_agent' in model and 'sales_agent' in teams:
    model = model.merge(teams, on='sales_agent', how='left', validate='many_to_one', suffixes=('', '_team'))
print('Joined shape:', model.shape)
display(model.head())

## Define target and predictors

Only decided opportunities are included. Outcome-derived fields and identifiers are excluded from predictors.

In [ ]:
model = model[model['deal_stage'].isin(['Won', 'Lost'])].copy()
model['target_won'] = (model['deal_stage'] == 'Won').astype('int8')
excluded = {'target_won','deal_stage','close_value','close_date','engage_date','opportunity_id','account','product_join_key'}
# Keep original descriptive product and sales-agent categories where available; exclude free-form identifiers.
feature_cols = [c for c in model.columns if c not in excluded and not c.endswith('_id')]
# Remove columns with a single observed value.
feature_cols = [c for c in feature_cols if model[c].nunique(dropna=True) > 1]
X = model[feature_cols].copy()
y = model['target_won'].copy()
# Convert date-like columns to strings only if they are retained; avoid using outcome timestamps by design.
for c in X.select_dtypes(include=['datetime','datetimetz']).columns:
    X[c] = X[c].astype(str)
print('Model rows:', len(model), '| Won rate:', round(y.mean(), 3))
print('Feature columns:', feature_cols)
X.head()

## Save the processed table

The processed file is a reproducible intermediate, not a replacement for the source data.

In [ ]:
out = ROOT / 'data' / 'processed'
out.mkdir(parents=True, exist_ok=True)
model.to_csv(out / 'modeling_dataset.csv', index=False)
print('Saved:', out / 'modeling_dataset.csv')